## Imports and Configuration

In [14]:
import pandas as pd
from sqlalchemy import create_engine, inspect
from urllib.parse import quote_plus

MYSQL_DB = {
    "user": "root",
    "pass": "Welcome123!",
    "host": "localhost",
    "port": "3306",
    "db": "classicmodels"
}

SS_DB = {
    "server": "localhost",
    "user": "sa",
    "pass": "Welcome@123!",
    "db": "mysql_db",
    "driver": "ODBC Driver 18 for SQL Server"
}

print("Libraries imported and config set")

Libraries imported and config set


## Create Engines and Test Connections
- verifies that both databases are reachable before you start moving data

In [15]:
# MySQL Engine - Note the 'mysql+pymysql' dialect
mysql_pass = quote_plus(MYSQL_DB['pass'])
mysql_url = f"mysql+pymysql://{MYSQL_DB['user']}:{mysql_pass}@{MYSQL_DB['host']}:{MYSQL_DB['port']}/{MYSQL_DB['db']}"
mysql_engine = create_engine(mysql_url)

# SQL Server Engine
ss_conn_str = (
    f"DRIVER={{{SS_DB['driver']}}};"
    f"SERVER={SS_DB['server']};"
    f"DATABASE={SS_DB['db']};"
    f"UID={SS_DB['user']};"
    f"PWD={SS_DB['pass']};"
    "Encrypt=yes;TrustServerCertificate=yes;"
)
ss_engine = create_engine(f"mssql+pyodbc:///?odbc_connect={ss_conn_str}")

# Test
try:
    with mysql_engine.connect() as conn:
        print("MySQL Connection Successful")
    with ss_engine.connect() as conn:
        print("SQL Server Connection Successful")
except Exception as e:
    print(f"Connection Error: {e}")

MySQL Connection Successful
SQL Server Connection Successful


In [16]:
inspector = inspect(mysql_engine)
mysql_tables = inspector.get_table_names()

print(f"Found {len(mysql_tables)} tables in SQL Server: {mysql_tables}")

Found 8 tables in SQL Server: ['customers', 'employees', 'offices', 'orderdetails', 'orders', 'payments', 'productlines', 'products']


## Inspect Tables
- discover all tables in MySQL Database

In [17]:
for table in mysql_tables:
    try:
        print(f"Reading {table}...")
        # Read from mysql database
        df = pd.read_sql_table(table, mysql_engine)

        if df.empty:
            print(f"Table {table} is empty. Skipping...")
            continue

        # append the prefix for the destination
        destination_name = f"stg_{table}"
        
        # Load to SQL Server(if_exists='replace' creates the DDL/Table automatically)
        print(f"Writing {table} to SQL Server ({len(df)} rows)...")
        df.to_sql(
            destination_name, 
            ss_engine, 
            if_exists='replace', 
            index=False, 
            chunksize=1000
        )

        print(f"Successfully moved {table} -> {destination_name}")
        
    except Exception as e:
        print(f"Failed to migrate {table}: {e}")

print("\n--- ALL TASKS COMPLETE ---")

Reading customers...
Writing customers to SQL Server (122 rows)...
Successfully moved customers -> stg_customers
Reading employees...
Writing employees to SQL Server (23 rows)...
Successfully moved employees -> stg_employees
Reading offices...
Writing offices to SQL Server (7 rows)...
Successfully moved offices -> stg_offices
Reading orderdetails...
Writing orderdetails to SQL Server (2996 rows)...
Successfully moved orderdetails -> stg_orderdetails
Reading orders...
Writing orders to SQL Server (326 rows)...
Successfully moved orders -> stg_orders
Reading payments...
Writing payments to SQL Server (273 rows)...
Successfully moved payments -> stg_payments
Reading productlines...
Writing productlines to SQL Server (7 rows)...
Successfully moved productlines -> stg_productlines
Reading products...
Writing products to SQL Server (110 rows)...
Successfully moved products -> stg_products

--- ALL TASKS COMPLETE ---
